# 07 — Modern Pipeline: EfficientNet-B0 Feature Extraction Stub & Full Plan

This notebook demonstrates that the modern pipeline scaffold is in place:
1. Frozen EfficientNet-B0 (ImageNet weights) extracts 1 280-dim pooled features.
2. A small proof-of-concept extraction on 16 sample images is saved to disk.
3. The full training and evaluation plan — including MC Dropout uncertainty quantification and temperature-scaling calibration — is documented below.

In [ ]:
import os
import sys
from pathlib import Path

_cwd = Path(os.getcwd())
for _root in [_cwd, *_cwd.parents]:
    if (_root / "skin_lesion" / "src" / "config.py").exists():
        sys.path.insert(0, str(_root / "skin_lesion" / "src"))
        break

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.models as tvm
from torchvision import transforms

from config import SEED, PROCESSED_DIR, MC_DROPOUT_T

torch.manual_seed(SEED)
DEVICE = (
    "cuda" if torch.cuda.is_available() else
    "mps"  if torch.backends.mps.is_available() else
    "cpu"
)
print(f"Device: {DEVICE}")

## 1 — Load 16 sample images from the balanced subset

In [ ]:
df = pd.read_csv(PROCESSED_DIR / "balanced_subset.csv")
df = df[df["image_path"].notna()]

# 8 melanoma + 8 nevus for a balanced sample
rng = np.random.default_rng(SEED)
def stratified_sample(df, n_per_class=8):
    parts = []
    for lbl in [0, 1]:
        sub = df[df["label"] == lbl]
        idx = rng.choice(len(sub), size=min(n_per_class, len(sub)), replace=False)
        parts.append(sub.iloc[idx])
    return pd.concat(parts).reset_index(drop=True)

sample_df = stratified_sample(df, n_per_class=8)
print(f"Sample: {len(sample_df)} images  "
      f"(melanoma={sample_df['label'].sum()}, nevus={(sample_df['label']==0).sum()})")

## 2 — Build the frozen EfficientNet-B0 feature extractor

EfficientNet-B0 ends with:
```
features  →  avgpool (adaptive, 1×1)  →  flatten  →  classifier (Dropout + Linear → 1000)
```
We replace `model.classifier` with `nn.Identity()` to obtain the 1 280-dim pre-logit representation.  
All backbone parameters are frozen (`requires_grad=False`) — only the classification head added later will be trained.

In [ ]:
weights = tvm.EfficientNet_B0_Weights.DEFAULT
backbone = tvm.efficientnet_b0(weights=weights)

# Freeze all backbone parameters
for param in backbone.parameters():
    param.requires_grad = False

# Remove the classifier head to expose 1280-dim pooled features
backbone.classifier = nn.Identity()
backbone = backbone.to(DEVICE).eval()

# Confirm: no trainable parameters in the backbone
n_trainable = sum(p.numel() for p in backbone.parameters() if p.requires_grad)
n_total     = sum(p.numel() for p in backbone.parameters())
print(f"Backbone parameters  : {n_total:,}")
print(f"Trainable parameters : {n_trainable}  (should be 0)")

## 3 — Extract features for the 16 sample images

In [ ]:
# ImageNet normalisation expected by EfficientNet-B0
preprocess = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

feats_list = []
labels_list = []
ids_list = []

with torch.no_grad():
    for _, row in sample_df.iterrows():
        img_bgr = cv2.imread(str(row["image_path"]))
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        tensor  = preprocess(img_rgb).unsqueeze(0).to(DEVICE)  # (1, 3, 224, 224)
        feat    = backbone(tensor).squeeze(0).cpu().numpy()     # (1280,)
        feats_list.append(feat)
        labels_list.append(int(row["label"]))
        ids_list.append(row["image_id"])

X_sample = np.stack(feats_list)          # (16, 1280)
y_sample = np.array(labels_list)         # (16,)
ids_sample = np.array(ids_list)

print(f"Feature matrix shape : {X_sample.shape}   ← (n_images, 1280)")
print(f"Label vector shape   : {y_sample.shape}")
print(f"Gradients on any feat tensor: False  (extracted under torch.no_grad())")

## 4 — Save proof-of-concept features

In [ ]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
out_path = PROCESSED_DIR / "effnet_sample.npz"
np.savez_compressed(out_path, X=X_sample, y=y_sample, image_ids=ids_sample)
print(f"Saved {X_sample.shape[0]} sample features to: {out_path}")

---
## 5 — Full Modern Pipeline Plan

### 5.1 — Classification head architecture

A lightweight trainable head is appended on top of the 1 280-dim frozen features.  
The Dropout layer is **required** for MC Dropout uncertainty estimation at inference time:

```python
class ClassificationHead(nn.Module):
    def __init__(self, in_features=1280, hidden=256, p_drop=0.3, n_classes=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, hidden),
            nn.ReLU(),
            nn.Dropout(p=p_drop),   # ← kept ACTIVE during MC Dropout inference
            nn.Linear(hidden, n_classes),
        )
    def forward(self, x):
        return self.net(x)
```

Training details:
- **Optimizer**: Adam, lr=1e-3, weight_decay=1e-4.
- **Loss**: cross-entropy with optional class weights to handle residual imbalance.
- **Early stopping**: monitor validation loss, patience=10 epochs.
- **Batch size**: 64; input images resized to 224×224 with ImageNet normalisation.

### 5.2 — Monte Carlo Dropout for uncertainty quantification

At test time the standard approach calls `model.eval()`, which deactivates Dropout.  
MC Dropout (Gal & Ghahramani, 2016) instead keeps Dropout **stochastic** during inference to obtain a distribution over predictions:

```python
def mc_dropout_predict(backbone, head, x, T=MC_DROPOUT_T):
    backbone.eval()       # backbone always frozen
    head.eval()           # set BN layers etc. to eval…
    # …but re-enable Dropout stochasticity:
    for m in head.modules():
        if isinstance(m, nn.Dropout):
            m.train()

    probs_list = []
    with torch.no_grad():
        feat = backbone(x)                   # extract features once
        for _ in range(T):
            logits = head(feat)              # T stochastic forward passes
            probs_list.append(torch.softmax(logits, dim=-1))

    probs_mc   = torch.stack(probs_list).mean(dim=0)   # (N, 2) — predictive mean
    entropy    = -(probs_mc * probs_mc.log()).sum(-1)   # (N,)   — predictive entropy
    return probs_mc, entropy
```

The **predictive entropy** H[p] = −Σ p_c · log(p_c) ranges from 0 (certain) to log(2) ≈ 0.693 (maximally uncertain).  
High-entropy samples will be visualised as a "gallery of hard cases" in the comparison notebook.

### 5.3 — Calibration via temperature scaling

After MC Dropout averaging, raw probabilities may still be miscalibrated.  
Temperature scaling (Guo et al., 2017) divides the logits by a scalar T_temp > 0 before softmax:

```
p̂ = softmax(logits / T_temp)
```

T_temp is found by minimising NLL on the **validation set only** (test set untouched):  
```python
from scipy.optimize import minimize_scalar
result = minimize_scalar(lambda t: nll(logits_val / t, y_val), bounds=(0.1, 10), method='bounded')
T_temp = result.x
```

After calibration, the reliability diagram (15 bins) and ECE are recomputed on the test set to show the before/after improvement.  
The calibrated probabilities are also used for the cost-sensitive decision rule.

### 5.4 — Classical vs modern comparison plan

| Metric | Classical (GMM + Bayes) | Modern (EfficientNet + MC Dropout) |
|--------|------------------------|------------------------------------|
| AUC-ROC | ✓ (notebook 06) | ✓ (to be added to calibration_summary.csv) |
| ECE (before/after calibration) | ✓ (notebook 06) | ✓ (before = raw MC mean; after = temperature-scaled) |
| Total cost (5·FN + 1·FP, θ=1/6) | ✓ (notebook 05) | ✓ (applied to MC-averaged + calibrated posteriors) |
| Uncertainty gallery | — | ✓ 8 highest-entropy test images (4 correct, 4 incorrect if available) |
| Reliability diagram | ✓ (notebook 06) | ✓ overlay both models in one figure for direct visual comparison |

The final comparison notebook will load `calibration_summary.csv` and append a row for the modern model,  
so all results are consolidated in one artefact.


---
## 6 — References

**MC Dropout on skin lesion data:**  
Singh, A., Bhatt, U., Ghassemi, M. (2020). *SkiNet: Uncertainty-Aware Skin Lesion Segmentation using Monte Carlo Dropout*. arXiv:2012.15049. Demonstrates MC Dropout for epistemic uncertainty estimation on ISIC-2018 dermoscopy images, directly motivating the approach adopted in this project.

**Original MC Dropout paper:**  
Gal, Y., & Ghahramani, Z. (2016). *Dropout as a Bayesian Approximation: Representing Model Uncertainty in Deep Learning*. Proceedings of the 33rd ICML, PMLR 48:1050–1059. Provides the theoretical justification for using stochastic Dropout at inference as an approximation to a deep Gaussian process posterior.

**Temperature scaling:**  
Guo, C., Pleiss, G., Sun, Y., & Weinberger, K. Q. (2017). *On Calibration of Modern Neural Networks*. Proceedings of the 34th ICML, PMLR 70:1321–1330. Introduces temperature scaling as a simple post-hoc calibration method and shows it consistently reduces ECE across architectures.